# 05 Sensitivity analyses: predictor sets and the WHO minimum indicator set (Tables S17, S18, Figure S3)

Saved hyperparameters and fitted pipelines in `models/` are reused when present; delete them to re-tune.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from IPython.display import display
import utils

In [ ]:
train = utils.load_object(utils.DATA + "/train_2012_2014")
validation = utils.load_object(utils.DATA + "/validation_2015")
test_local = utils.load_object(utils.DATA + "/test_local_2016_moh")
y_train = train["OBITO"]
y_val = validation["OBITO"].values
y_local = test_local["OBITO"].values

base = ["IDADEMAE", "LOCNASC", "IDANOMAL", "PARTO"]
predictor_sets = {"Base (no timing)": base,
                  "Base + Apgar-1": base + ["APGAR1"],
                  "Base + Apgar-5": base + ["APGAR5"],
                  "Base + birth weight": base + ["PESO"],
                  "Base + gestational age": base + ["SEMAGESTAC"]}
{name: len(features) for name, features in predictor_sets.items()}

## Table S17: metric battery by predictor set, one sheet per model (local test set)

In [ ]:
# for each predictor set: tune the boosters (reuse if saved), fit, choose thresholds on the validation
# cohort, and score on the local test set with confidence intervals
battery_by_set = {label: {} for _, label in utils.MODELS_ORDER}
for set_name, features in predictor_sets.items():
    continuous = [c for c in utils.CONTINUOUS if c in features]
    categorical = [c for c in utils.CATEGORICAL if c in features]
    for model_name, label in utils.MODELS_ORDER:
        tag = model_name + "__" + set_name.replace(" ", "_").replace("+", "plus")
        model_path = utils.MODELS + "/predictor_set_" + tag + ".pkl"
        if os.path.exists(model_path):
            pipeline = utils.load_object(model_path)
        else:
            if model_name == "logistic_regression":
                params = {}
            else:
                params_path = utils.MODELS + "/predictor_set_params_" + tag + ".pkl"
                if os.path.exists(params_path):
                    params = utils.load_object(params_path)
                else:
                    params = utils.tune_hyperparameters(model_name, train[features], y_train, continuous, categorical,
                                                        n_evaluations=100, n_folds=10)["best_parameters"]
                    utils.save_object(params, params_path)
            pipeline = utils.make_pipeline(model_name, params, continuous, categorical)
            pipeline.fit(train[features], y_train)
            utils.save_object(pipeline, model_path)
        points = utils.operating_points(y_val, pipeline.predict_proba(validation[features])[:, 1])
        set_thresholds = dict(zip(points.operating_point, points.threshold))
        p_local = pipeline.predict_proba(test_local[features])[:, 1]
        battery_by_set[label][set_name] = utils.evaluate_with_confidence_intervals(y_local, p_local, set_thresholds)
    print("predictor set done:", set_name)

In [ ]:
sheets_s17 = {}
for model_name, label in utils.MODELS_ORDER:
    sheets_s17[label] = utils.metric_battery_wide({set_name: battery_by_set[label][set_name] for set_name in predictor_sets})
utils.save_sheets(sheets_s17, utils.SUPPLEMENT_TABLES, "Table_S17")
for sheet, table in sheets_s17.items():
    print(sheet)
    display(table)

## Figure S3: AUROC by predictor set

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
set_order = list(predictor_sets)
auroc_rows = []
for model_name, label in utils.MODELS_ORDER:
    aurocs = []
    for set_name in set_order:
        row = battery_by_set[label][set_name]
        aurocs.append(float(row.loc[row.metric == "auroc", "value"].iloc[0]))
    ax.plot(set_order, aurocs, marker="o", color=utils.MODEL_COLORS[label], label=label)
    auroc_rows.append(pd.DataFrame({"model": label, "predictor_set": set_order, "auroc": aurocs}))
ax.set_xlabel("Predictor set"); ax.set_ylabel("AUROC (local 2016)")
ax.tick_params(axis="x", rotation=30); ax.legend()
utils.save_figure(fig, utils.SUPPLEMENT_FIGURES, "Figure_S3_predictor_set_AUROC",
                  data=pd.concat(auroc_rows, ignore_index=True))
fig

## Table S18: WHO minimum indicator set versus the full eight predictors (local test set)

In [ ]:
who_minimum = [c for c in utils.FEATURES if c not in ["APGAR1", "APGAR5", "IDANOMAL"]]
who_continuous = [c for c in utils.CONTINUOUS if c in who_minimum]
who_categorical = [c for c in utils.CATEGORICAL if c in who_minimum]

rows = []
for model_name, label in utils.MODELS_ORDER:
    who_path = utils.MODELS + "/who_minimum_" + model_name + ".pkl"
    if os.path.exists(who_path):
        who_model = utils.load_object(who_path)
    else:
        if model_name == "logistic_regression":
            params = {}
        else:
            params_path = utils.MODELS + "/who_minimum_params_" + model_name + ".pkl"
            if os.path.exists(params_path):
                params = utils.load_object(params_path)
            else:
                params = utils.tune_hyperparameters(model_name, train[who_minimum], y_train, who_continuous, who_categorical,
                                                    n_evaluations=100, n_folds=10)["best_parameters"]
                utils.save_object(params, params_path)
        who_model = utils.make_pipeline(model_name, params, who_continuous, who_categorical)
        who_model.fit(train[who_minimum], y_train)
        utils.save_object(who_model, who_path)
    who_probability = who_model.predict_proba(test_local[who_minimum])[:, 1]
    who_evaluation = utils.evaluate_with_confidence_intervals(y_local, who_probability, {})
    who_row = who_evaluation.loc[who_evaluation.metric == "auroc"].iloc[0]

    full_model = utils.load_object(utils.MODELS + "/model_" + model_name)
    full_auroc = roc_auc_score(y_local, full_model.predict_proba(test_local[utils.FEATURES])[:, 1])
    rows.append({"Model": label,
                 "WHO-minimum AUROC [95% CI]": utils.estimate_with_ci(who_row["value"], who_row["ci_low"], who_row["ci_high"]),
                 "Full 8-predictor AUROC": round(full_auroc, 3),
                 "AUROC gain from the three extra predictors": round(full_auroc - who_row["value"], 3)})
table_s18 = pd.DataFrame(rows)
utils.save_table(table_s18, utils.SUPPLEMENT_TABLES, "Table_S18")
table_s18